# Module 26: Capstone Payment Gateway AI Fraud — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/payment_gateway_platform.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import payment_gateway_platform

classes = [n for n, o in inspect.getmembers(payment_gateway_platform, inspect.isclass)
           if o.__module__ == 'payment_gateway_platform']
functions = [n for n, o in inspect.getmembers(payment_gateway_platform, inspect.isfunction)
             if o.__module__ == 'payment_gateway_platform']

print('module   : payment_gateway_platform')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(payment_gateway_platform, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Double entry ledger balanced postings

This is the module's own `test_double_entry_ledger_balanced_postings` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import pytest
from payment_gateway_platform import (
    AccountingDiscrepancyException,
    DoubleEntryLedger,
    EntryType,
    LedgerPosting,
    PaymentGatewayPlatform,
)

ledger = DoubleEntryLedger()
postings = [
    LedgerPosting("acct_alice", EntryType.DEBIT, 1000),
    LedgerPosting("acct_bob", EntryType.CREDIT, 1000),
]
tx = ledger.record_transaction("tx_1", postings, "Test Transfer")
assert tx.tx_id == "tx_1"
assert ledger.get_balance("acct_alice") == 1000
assert ledger.get_balance("acct_bob") == -1000
assert ledger.verify_ledger_integrity() is True

print('PASSED: test_double_entry_ledger_balanced_postings')

## 3. 🔮 Prediction — commit before you run

A payment is authorised but the capture call times out. Predict whether the customer was charged, and what mechanism lets you find out safely.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_double_entry_ledger_unbalanced_discrepancy`, which tests exactly this property.


In [ ]:
ledger = DoubleEntryLedger()
# Unbalanced: 1000 debit vs 800 credit
postings = [
    LedgerPosting("acct_alice", EntryType.DEBIT, 1000),
    LedgerPosting("acct_bob", EntryType.CREDIT, 800),
]
with pytest.raises(AccountingDiscrepancyException):
    ledger.record_transaction("tx_unbalanced", postings)

print('PASSED: test_double_entry_ledger_unbalanced_discrepancy')

## 4. Measure it: Payment authorization and fee math

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_payment_authorization_and_fee_math` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

gateway = PaymentGatewayPlatform(fee_percentage=0.029, fixed_fee_cents=30)

# $100.00 transaction -> Fee: 2.9% * 10000 = 290 + 30 = 320c ($3.20)
# Merchant payout: 10000 - 320 = 9680c ($96.80)
res = gateway.process_payment(
    idempotency_key="key_001",
    customer_id="alice",
    merchant_id="merchant_x",
    amount_cents=10000,
)

assert res["status"] == "AUTHORIZED"
assert res["amount_cents"] == 10000
assert res["fee_cents"] == 320
assert res["merchant_net_cents"] == 9680
assert res["is_idempotent_replay"] is False
assert len(gateway.outbox) == 1
assert gateway.ledger.verify_ledger_integrity() is True

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_payment_authorization_and_fee_math')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(payment_gateway_platform) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Idempotency keys are what make a timed-out payment safe to retry.
2. Authorise and capture are separate steps for a reason - model both.
3. A fraud score is an input to a decision, never the decision itself.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
